In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import pandas as pd

import torch
from torch.nn import CrossEntropyLoss,Linear
from torch.cuda.amp import GradScaler,autocast
from torch.utils.data import DataLoader,Dataset
from torchvision.utils import make_grid
import torchvision.transforms.v2 as v2
from torchvision.models import resnet50,ResNet50_Weights
from torch.optim import AdamW
from PIL import Image
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [ ]:
IMG_SIZE = 224

NUM_CLASSES = 2

EPOCHS = 20
BATCH_SIZE = 64

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3

PATIENCE = 3

MEAN_NORM = [0.485, 0.456, 0.406]
STD_NORM = [0.229, 0.224, 0.225]

NUM_WORKERS = 0
PIN_MEMORY = True

LABELS = ['Bird',"Drone"]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
class DBDataset(Dataset):
    def __init__(self,df,transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_path = self.df.iloc[index,0]
        label = self.df.iloc[index,1]

        img = Image.open(img_path).convert('RGB')

        if self.transforms:
            img = self.transforms(img)

        return img,label


In [ ]:
dataset_path = '/kaggle/input/datasets/abdullahbakr7/drones-vs-birds/dataset'

def prepare_dataframe(dataset_path):
    all_imgs = []
    all_labels = []

    for class_folder in os.listdir(dataset_path):
        class_idx = LABELS.index(class_folder.capitalize())
        class_path = os.path.join(dataset_path,class_folder)
        for img_item in os.listdir(class_path):
            all_imgs.append(os.path.join(class_path,img_item))
            all_labels.append(class_idx)

    df = pd.DataFrame({'Image Path': all_imgs,
                    'Image Label': all_labels})
    return df

df = prepare_dataframe(dataset_path)
df

In [ ]:
train_df , val_df = train_test_split(df,test_size=0.3,stratify=df['Image Label'],random_state=42)
val_df , test_df = train_test_split(val_df,test_size=0.15,stratify=val_df['Image Label'],random_state=42)

print('Train Length: ', len(train_df))
print('Validation Length: ', len(val_df))
print('Test Length: ', len(test_df))
print('Total: ', len(train_df) + len(val_df) + len(test_df))

In [ ]:
train_df['Image Label'].value_counts()

In [ ]:
val_df['Image Label'].value_counts()

In [ ]:
test_df['Image Label'].value_counts()

In [ ]:
def show_sample(df):
    _, ax = plt.subplots(4,4,figsize=(12,12))
    ax = ax.flatten()
    random_images = np.random.choice(df['Image Path'].values,16)
    for i in range(len(random_images)):
        img_path = random_images[i]
        ax[i].imshow(plt.imread(img_path))
        ax[i].axis('off')
        ax[i].set_title(f'Label: {img_path.split('/')[-2]}')
        
    plt.tight_layout()

show_sample(train_df)

In [ ]:
show_sample(val_df)

In [ ]:
show_sample(test_df)

In [ ]:
def get_transforms(size=IMG_SIZE, apply_on_train=False):
    """
    Build a transform pipeline for training or validation/test.

    Args:
        size: target square image size in pixels
        apply_on_train: if True, adds augmentation steps before normalization

    Returns:
        v2.Compose transform pipeline
    """
    base = [
        v2.Resize((size, size)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]

    augmentation = [
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(45),
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        v2.GaussianBlur(kernel_size=3),
        v2.RandomPerspective(p=0.2),
    ]

    # ImageNet mean/std normalization (required for pretrained models)
    tail = [v2.Normalize(MEAN_NORM, STD_NORM)]

    if apply_on_train:
        return v2.Compose(base + augmentation + tail)
    return v2.Compose(base + tail)

In [ ]:
train_ds = DBDataset(train_df,get_transforms(IMG_SIZE,True))
val_ds = DBDataset(val_df,get_transforms(IMG_SIZE))
test_ds = DBDataset(test_df,get_transforms(IMG_SIZE))

print('Total: ', len(train_ds) + len(val_ds) + len(test_ds))

In [ ]:
train_dl = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
val_dl = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
test_dl = DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)